In [ ]:
--@exclude_input=xyf_bi_dev.eventid_median_0616
--@exclude_input=xyf_dwd.dwd_event_tracking_log_di
--@exclude_input=xyf_bi_dev.utm_source_channel_v1_cdf_v
--@exclude_input=xyf_dwd.dwd_inloan_t_decision_result_detail_df
--odps sql 
--********************************************************************--
--author:邵逸婷
--create time:2025-03-27 11:38:53
--********************************************************************--

DROP TABLE IF EXISTS bi_app_zhuanhua_shouxin_uid
;

CREATE TABLE bi_app_zhuanhua_shouxin_uid AS
SELECT  shouxin.user_no
        ,date(shouxin.created_time) dt
        ,zhou.week_range wt
        ,SUBSTR(shouxin.created_time,1,7) mt
        ,case when upper(qu_zhu.source) like '%HRUI01%' then '其他付费-华瑞01'
        when upper(qu_zhu.source) like '%HRUI02%' then '其他付费-华瑞02' 
        else  qu_zhu.first_channel end as  biz_type_bi
        ,qu_zhu.second_channel as biz_type_bi_2
        ,qu_gui.first_channel AS biz_type_bi_3 
        ,qu_gui.second_channel as biz_type_bi_4
        ,补件页是否可曝光 
        ,case when shouxin.inner_app = 'xyf01' then 'APP全流程' else 'API半流程' end as 是否半流程
        ,count(distinct case when po.cust_no is not null then shouxin.user_no end)是否通过预借款发起
        ,COUNT(DISTINCT CASE    WHEN shouxin.是否小程序授信 = '小程序授信' THEN shouxin.user_no END) AS 是否小程序授信
        ,COUNT(DISTINCT CASE    WHEN shouxin.is_首次授信成功 = 1 THEN shouxin.user_no END) AS 是否首次APP授信成功
        ,MAX(shouxin.init_credit_line / 100) AS 授信金额
        ,CASE   WHEN MAX(shouxin.init_credit_line / 100) BETWEEN 1 AND 3000 THEN '1. 0k-3k'
                WHEN MAX(shouxin.init_credit_line / 100) BETWEEN 3001 AND 5000 THEN '2. 3k-5k'
                WHEN MAX(shouxin.init_credit_line / 100) BETWEEN 5001 AND 10000 THEN '3.5k-1w'
                WHEN MAX(shouxin.init_credit_line / 100) BETWEEN 10001 AND 20000 THEN '4.1w-2w'
                WHEN MAX(shouxin.init_credit_line / 100) BETWEEN 20001 AND 50000 THEN '5.2w-5w'
                WHEN MAX(shouxin.init_credit_line / 100) BETWEEN 50001 AND 100000 THEN '6.5w-10w'
                WHEN MAX(shouxin.init_credit_line / 100) > 100000 THEN '7.10w+'
                ELSE ''
        END 授信金额_level
        ,CASE   WHEN MAX(cast(c.last_app_activation_score as DECIMAL(18,3))) >= 8 THEN '8+'
                WHEN MAX(cast(c.last_app_activation_score as DECIMAL(18,3))) > 0 AND MAX(cast(c.last_app_activation_score as DECIMAL(18,3))) <= 2 THEN 1
                WHEN MAX(cast(c.last_app_activation_score as DECIMAL(18,3))) > 2 AND MAX(cast(c.last_app_activation_score as DECIMAL(18,3))) < 8 THEN FLOOR(MAX(cast(c.last_app_activation_score as DECIMAL(18,3))))
                ELSE ''
        END AS 实际执行人行完件评级_v3_v4
        ,MAX(cast(c.last_app_activation_score as DECIMAL(18,3))) AS last_app_activation_score
        ,MIN(c.register_time) AS register_time
        ,CASE   WHEN DATEDIFF(date(shouxin.created_time),MIN(c.register_time)) = 0 THEN '0d'
                WHEN DATEDIFF(date(shouxin.created_time),MIN(c.register_time)) BETWEEN 1 AND 30 THEN '1-30d'
                WHEN DATEDIFF(date(shouxin.created_time),MIN(c.register_time)) BETWEEN 31 AND 60 THEN '31d-60d'
                WHEN DATEDIFF(date(shouxin.created_time),MIN(c.register_time)) > 60 THEN '61d+'
                ELSE ''
        END AS 注册授信时间差
        ,count(distinct case when datediff(c.first_login_time,shouxin.created_time) <= 0 then shouxin.user_no end)T0登录
        ,count(distinct case when (datediff(m.event_time,shouxin.created_time) =0   or DATEDIFF(t.event_time,shouxin.created_time) = 0 )then shouxin.user_no END )进入提现页----进入提现页,点击借款按钮,需要补件用户数,需要补联系人用户,需要补OCR用户,需要补银行卡用户,补充联系人,补充OCR,补充银行卡用户
        ,count(distinct case when (datediff(m1.event_time,shouxin.created_time) = 0 or DATEDIFF(t1.event_time,shouxin.created_time) = 0)then shouxin.user_no end)点击借款按钮
        ,count(distinct case when datediff(m2.event_time,shouxin.created_time) = 0 then shouxin.user_no end)进入补件页人数----
        ,count(distinct case when (datediff(m1.event_time,shouxin.created_time) = 0 or DATEDIFF(t1.event_time,shouxin.created_time) = 0) 
        and (bujian.首次填写联系人时间<=m1.event_time or bujian.首次填写联系人时间<=t1.event_time) 
        and  (bujian.首次填写银行卡时间<=m1.event_time or bujian.首次填写银行卡时间<=t1.event_time )
        and (bujian.首次完成OCR时间<=m1.event_time or bujian.首次完成OCR时间<=t1.event_time)
         and (bujian.首次完成基本信息时间<=m1.event_time or bujian.首次完成基本信息时间<=t1.event_time)
         and (bujian.首次完成人脸时间<=m1.event_time or bujian.首次完成人脸时间<=t1.event_time) then shouxin.user_no end)点击借款按钮_无需补件
        ,count(distinct case when (datediff(m1.event_time,shouxin.created_time) = 0 or DATEDIFF(t1.event_time,shouxin.created_time) = 0) and datediff(bujian.首次填写联系人时间，shouxin.created_time) <= 0 and datediff(bujian.首次填写银行卡时间,shouxin.created_time) <= 0
        and datediff(bujian.首次完成OCR时间,shouxin.created_time) <= 0  and datediff(bujian.首次完成人脸时间,shouxin.created_time)<=0 
        and datediff(bujian.首次完成基本信息时间,shouxin.created_time)<=0 then shouxin.user_no end)点击借款按钮_不缺三要素
        ---完成补件人数
        ,count(distinct case when datediff(m2.event_time,shouxin.created_time) = 0 and datediff(bujian.首次填写联系人时间，shouxin.created_time) <= 0 and datediff(bujian.首次填写银行卡时间,shouxin.created_time) <= 0
        and datediff(bujian.首次完成OCR时间,shouxin.created_time) <= 0  and datediff(bujian.首次完成人脸时间,shouxin.created_time)<=0 and datediff(bujian.首次完成基本信息时间,shouxin.created_time)<=0 then shouxin.user_no end)进入补件页完成补件人数
        ,count(distinct case when bujian.用户在授信前是否填写联系人 = 0 or 用户在授信前是否完成OCR = 0 or 用户在授信前是否完成银行卡 = 0 or 用户在授信前是否完成人脸 = 0 or 用户在授信前是否完成基本信息 = 0
        then shouxin.user_no end )需要补件用户数
        ,count(distinct case when bujian.用户在授信前是否填写联系人 = 0 then shouxin.user_no end)需要补联系人用户
        ,count(distinct case when bujian.用户在授信前是否完成OCR = 0 then shouxin.user_no end)需要补OCR用户
        ,count(distinct case when bujian.用户在授信前是否完成银行卡 = 0 then shouxin.user_no end)需要补银行卡用户
        ,count(distinct case when bujian.用户在授信前是否填写联系人 = 0  and datediff(bujian.首次填写联系人时间，shouxin.created_time) = 0 then shouxin.user_no end)补充联系人
        ,count(distinct case when bujian.用户在授信前是否完成OCR = 0 and datediff(bujian.首次完成OCR时间,shouxin.created_time) = 0 then shouxin.user_no end)补充OCR
        ,count(distinct case when bujian.用户在授信前是否完成银行卡 = 0  and datediff(bujian.首次填写银行卡时间,shouxin.created_time) = 0 then shouxin.user_no end)补充银行卡用户
        ,count(distinct case when (bujian.用户在授信前是否填写联系人 = 0 or 用户在授信前是否完成OCR = 0 or 用户在授信前是否完成银行卡 = 0 or 用户在授信前是否完成人脸 = 0 or 用户在授信前是否完成基本信息 = 0)
        and  datediff(bujian.首次填写联系人时间，shouxin.created_time) <= 0  and datediff(bujian.首次完成OCR时间,shouxin.created_time) <= 0
        and datediff(bujian.首次填写银行卡时间,shouxin.created_time) <= 0 and datediff(bujian.首次完成人脸时间,shouxin.created_time)<=0 and datediff(bujian.首次完成基本信息时间,shouxin.created_time)<=0 then shouxin.user_no end)完成所有补件用户 
        ,COUNT(DISTINCT 
              CASE    WHEN DATEDIFF(o.first_order_time,shouxin.created_time) = 0 THEN shouxin.user_no END
        ) 提现人
        ,COUNT(DISTINCT 
              CASE    WHEN DATEDIFF(o.first_order_time,shouxin.created_time) BETWEEN 0 AND 3 THEN shouxin.user_no END
        ) T3提现人
        ,COUNT(DISTINCT 
              CASE    WHEN DATEDIFF(o.first_order_time,shouxin.created_time) BETWEEN 0 AND 7 THEN shouxin.user_no END
        ) T7提现人
        ,COUNT(DISTINCT 
              CASE    WHEN DATEDIFF(o.first_order_time,shouxin.created_time) BETWEEN 0 AND 30 THEN shouxin.user_no END
        ) T30提现人
        ,MAX(CASE    WHEN DATEDIFF(o.first_order_time,shouxin.created_time) = 0 THEN o.loan_amt END) 提现金额
        ,MAX(
            CASE    WHEN DATEDIFF(o.first_order_time,shouxin.created_time) BETWEEN 0 AND 7 THEN o.loan_amt END
        ) 提现金额7
        ,MAX(
            CASE    WHEN DATEDIFF(o.first_order_time,shouxin.created_time) = 0 THEN o.loan_amt * o.period END
        ) 提现金额_period
        ,MAX(
            CASE    WHEN DATEDIFF(o.first_order_time,shouxin.created_time) = 0 AND o.loan_time IS NOT NULL THEN o.loan_amt END
        ) 放款金额
        ,MAX(
            CASE    WHEN DATEDIFF(o.first_order_time,shouxin.created_time) BETWEEN 0 AND 3 AND o.loan_time IS NOT NULL THEN o.loan_amt END
        ) T3放款金额
        ,MAX(
            CASE    WHEN DATEDIFF(o.first_order_time,shouxin.created_time) BETWEEN 0 AND 7 AND o.loan_time IS NOT NULL THEN o.loan_amt END
        ) T7放款金额
        ,MAX(
            CASE    WHEN DATEDIFF(o.first_order_time,shouxin.created_time) BETWEEN 0 AND 30 AND o.loan_time IS NOT NULL THEN o.loan_amt END
        ) T30放款金额
        ,MAX(
            CASE    WHEN DATEDIFF(o.first_order_time,shouxin.created_time) = 0 AND o.loan_time IS NOT NULL THEN o.loan_amt * o.period END
        ) 放款金额_period
        ,COUNT(DISTINCT 
              CASE    WHEN DATEDIFF(o.first_order_time,shouxin.created_time) = 0 AND o.risk_status = 'pass' THEN shouxin.user_no END
        ) 风控通过人
        ,COUNT(DISTINCT 
              CASE    WHEN DATEDIFF(o.first_order_time,shouxin.created_time) BETWEEN 0 AND 3 AND o.risk_status = 'pass' THEN shouxin.user_no END
        ) T3风控通过人
        ,COUNT(DISTINCT 
              CASE    WHEN DATEDIFF(o.first_order_time,shouxin.created_time) BETWEEN 0 AND 7 AND o.risk_status = 'pass' THEN shouxin.user_no END
        ) T7风控通过人
        ,max(CASE    WHEN DATEDIFF(o.first_order_time,shouxin.created_time) = 0 AND o.risk_status = 'pass' THEN o.loan_amt END)风控通过金额 ----风控通过金额,T7风控通过金额
        ,max(CASE    WHEN DATEDIFF(o.first_order_time,shouxin.created_time) BETWEEN 0 AND 7 AND o.risk_status = 'pass' THEN o.loan_amt END)T7风控通过金额
        ,COUNT(DISTINCT 
              CASE    WHEN DATEDIFF(o.first_order_time,shouxin.created_time) = 0 AND o.loan_time IS NOT NULL THEN shouxin.user_no END
        ) 放款人
        ,COUNT(DISTINCT 
              CASE    WHEN DATEDIFF(o.first_order_time,shouxin.created_time) BETWEEN 0 AND 3 AND o.loan_time IS NOT NULL THEN shouxin.user_no END
        ) T3放款人
        ,COUNT(DISTINCT 
              CASE    WHEN DATEDIFF(o.first_order_time,shouxin.created_time) BETWEEN 0 AND 7 AND o.loan_time IS NOT NULL THEN shouxin.user_no END
        ) T7放款人
        ,COUNT(DISTINCT 
              CASE    WHEN DATEDIFF(o.first_order_time,shouxin.created_time) BETWEEN 0 AND 30 AND o.loan_time IS NOT NULL THEN shouxin.user_no END
        ) T30放款人
        ,MAX(
            CASE    WHEN DATEDIFF(o.first_order_time,shouxin.created_time) = 0 AND o.loan_time IS NOT NULL THEN shouxin.init_credit_line / 100 END
        ) 授信额度_放款
        ,MAX(
            CASE    WHEN DATEDIFF(o.first_order_time,shouxin.created_time) BETWEEN 0 AND 7 AND o.loan_time IS NOT NULL THEN shouxin.init_credit_line / 100 END
        ) 授信额度_放款7
        ,MAX(
            CASE    WHEN DATEDIFF(o.first_order_time,shouxin.created_time) BETWEEN 0 AND 30 AND o.loan_time IS NOT NULL THEN shouxin.init_credit_line / 100 END
        ) 授信额度_放款30 ----短信渠道商也不需要
FROM    (
            SELECT  *
                    ,ROW_NUMBER() OVER (PARTITION BY cust_no ORDER BY credit_success_time ) AS is_首次授信成功
                    ,CASE   WHEN client_code IN ('MPP001000068') THEN '小程序授信'
                            ELSE '非小程序授信'
                    END AS 是否小程序授信
                    ,case when date(created_time)>='2025-05-07' and RANDOMV3('pgc_daizhong_ab',user_no,3) between 0 and 499 then 'groupB_补件页曝光组'
                    else '补件页不可曝光组'
                    end as 补件页是否可曝光 
            FROM    xyf_dwd.dwd_preloan_credit_apply_df
            WHERE   pt = '${bizdate}'
            AND     date(created_time) >= '2024-01-01'
            AND     app IN ('xyf01')
            AND     status = 2 --成功
            AND     inner_app IN ( 'xyf01','xyf01_hrui02','xyf01_xcjr','xyf01_hrui01','xyf01_alyxy','xyf01_alygd','xyf01_alyfz','xyf01_zyxj01', 'xyf01_zyxjwld01','xyf01_zyxjzl01','xyf01_elm')
            and     nvl(app_activation_type,'') <> 'loan_recredit_activation'
            AND     biz_flow_number NOT IN (
                        -- 虚假给额的授信成功用户口径，biz_flow_number关联授信表
                        SELECT  biz_flow_number -- 授信biz_flow_number
                        -- ,id_card_number
                        -- ,GET_JSON_OBJECT(context,"$.app_new_risk_mark_output") AS 人群类型
                        FROM    xyf_dwd.dwd_inloan_t_decision_result_detail_df
                        WHERE   pt = MAX_PT('xyf_dwd.dwd_inloan_t_decision_result_detail_df')
                        AND     enginecode = 'jcl_20240923000003' -- AND     GET_JSON_OBJECT(context,"$.app_new_risk_mark_output") = 'fake_activation'
                        AND     GET_JSON_OBJECT(context,"$.app_new_risk_mark_output") RLIKE 'fake_activation'
                        AND     decision_time >= '2024-10-15 00:00:00' --
                        -- 存在少量异常数据，同一个biz_flow_number+enginename 存在多条记录；下面排序做个兜底的清洗
                        QUALIFY ROW_NUMBER() OVER (PARTITION BY biz_flow_number,date(decision_time) ORDER BY decision_time DESC ) = 1
                    ) 
        ) shouxin

 left anti join (
             SELECT  DISTINCT biz_flow_number
                FROM    xyf_dwd.dwd_inloan_t_decision_result_detail_df
                WHERE   enginecode = 'jcl_20240722000003'
                AND     pt = MAX_PT('xyf_dwd.dwd_inloan_t_decision_result_detail_df')
                AND     inner_app <> 'xyf01_test1'
        )b 
        on shouxin.biz_flow_number = b.biz_flow_number
        
LEFT  JOIN   (
                --归因渠道相关--人行评级ping2，v2人行完件评级，是不是可以用last_activation_success_score
                SELECT  c.*
                        ,CASE   WHEN a.attribution_source IS NULL THEN c.current_utm_source
                                ELSE a.attribution_source
                        END AS attribution_source
                FROM    (
                            SELECT  *
                            FROM    xyf_dws.dws_preloan_register_conversion_df
                            WHERE   pt = MAX_PT('xyf_dws.dws_preloan_register_conversion_df')
                            AND     app IN ('xyf','xyf01')
                        ) c
                LEFT JOIN   (
                                SELECT  user_id
                                        ,attribution_source
                                FROM    xyf_dwd.dwd_xyf_flow_sys_flow_attribution_result_dup_df
                                WHERE   pt = '${bizdate}'
                                QUALIFY ROW_NUMBER() OVER (PARTITION BY user_id ORDER BY created_time DESC ) = 1
                            ) a
                ON      c.app_user_id = a.user_id
            ) c
ON      shouxin.user_no = c.app_user_id
--c表不用cust_no想连是因为连了渠道，同一个cust——no对于的不同user_no可以属于不同的渠道

left join  (
                            SELECT  *
                            FROM    xyf_dws.dws_preloan_register_conversion_df
                            WHERE   pt = MAX_PT('xyf_dws.dws_preloan_register_conversion_df')
                            AND     app IN ('xyf','xyf01')
                        ) c1 
on shouxin.cust_no = c1.cust_no
--因为会出现用户使用另一个user——no进入提现页的情况，所以用cust_noz转成app_user_id相连
LEFT JOIN   (
                -- SELECT  date(created_time)
                --         ,credit_user_id
                --         ,mobile
                --         ,app
                --         ,MIN(created_time) AS event_time
                -- FROM    xyf_dwd.dwd_biz_event_report_di
                -- WHERE   pt >= '20240101'
                -- and     pt <= '20250616'
                -- AND     event_id = '1003806'
                -- AND     app IN ('xyf','xyf01')
                -- AND     credit_user_id IS NOT NULL
                -- GROUP BY date(created_time)
                --          ,credit_user_id
                --          ,mobile
                --          ,app

                 select *
                from xyf_bi_dev.eventid_median_0616
                where event_id = '1003806'
            ) m--提现页
ON      c1.mobile = m.mobile --and date(shouxin.created_time) = to_date(m.event_time)
and     datediff(to_date(m.event_time),shouxin.created_time) between 0 and 7 
LEFT JOIN   (
                -- SELECT  date(created_time)
                --         ,credit_user_id
                --         ,mobile
                --         ,app
                --         ,MIN(created_time) AS event_time
                -- FROM    xyf_dwd.dwd_biz_event_report_di
                -- WHERE   pt >= '20240101'
                -- and     pt <= '20250616'
                -- AND     event_id = '1003810'
                -- AND     app IN ('xyf','xyf01')
                -- AND     credit_user_id IS NOT NULL
                -- GROUP BY date(created_time)
                --          ,credit_user_id
                --          ,mobile
                --          ,app
                
                 select *
                from xyf_bi_dev.eventid_median_0616
                where event_id = '1003810'
            ) m1--点击借款按钮
ON      c1.mobile = m1.mobile
and     datediff(to_date(m1.event_time),shouxin.created_time) between 0 and 7 
left join (--提现页新埋点
    
              SELECT  date(tracking_timestamp) dt,user_no,min(tracking_timestamp) event_time
                FROM    xyf_dwd.dwd_event_tracking_log_di
                WHERE   pt >= '20250616'
                AND     tracking_id IN ('JK-0-0-0-3','JK-110-0-0-1272')
                GROUP BY dt,user_no
)t  
on t.user_no = c1.app_user_id
and datediff(t.event_time,shouxin.created_time) between 0 and 7
left join (--点击借款按钮新埋点
    
              SELECT  date(tracking_timestamp) dt,user_no,min(tracking_timestamp) event_time
                FROM    xyf_dwd.dwd_event_tracking_log_di
                WHERE   pt >= '20250616'
                AND     tracking_id IN ('JK-52-0-201-287')
                GROUP BY dt,user_no
)t1  
on t1.user_no = c1.app_user_id
and datediff(t1.event_time,shouxin.created_time) between 0 and 7
left join (--展示补件埋点

-- SELECT  date(created_time) created_date
--                         ,credit_user_id
--                         ,mobile
--                       ,name
--                         ,MIN(created_time) AS event_time
--                 FROM    xyf_dwd.dwd_biz_event_report_di
--                 WHERE   pt >= '20250526'
--                 AND     event_id = '1005221'
--                 AND     app IN ('xyf','xyf01')
--                 AND     credit_user_id IS NOT NULL
--                 GROUP BY date(created_time)
--                          ,credit_user_id
--                          ,mobile
--                          ,app
--                          ,name


                         select date(tracking_timestamp) created_date
                         ,user_no
                         ,min(tracking_timestamp) as event_time
                          from xyf_dwd.dwd_event_tracking_log_di
                 where pt >='20250508'
                 and tracking_id = 'JK-64-0-0-661'
                 group by user_no,created_date
)m2---展示补件
--on c.mobile = m2.mobile  
--on shouxin.user_no = m2.user_no
on c1.app_user_id = m2.user_no
and  datediff(to_date(m2.event_time),shouxin.created_time) between 0 and 7 
left join (
    select *
    from xyf_bi_dev.bujian_syt
)bujian 
on bujian.cust_no = shouxin.cust_no
and to_date(bujian.created_time) = date(shouxin.created_time)

LEFT JOIN   (
                SELECT  source
                        ,zero_channel
                        ,first_channel
                        ,second_channel
                FROM    xyf_bi_dev.utm_source_channel_v1_cdf_v
                WHERE   pt = MAX_PT('xyf_bi_dev.utm_source_channel_v1_cdf_v')
                GROUP BY source
                         ,zero_channel
                         ,first_channel
                         ,second_channel
            ) qu_gui
ON      qu_gui.source = c.attribution_source
LEFT JOIN   (
                SELECT  source
                        ,zero_channel
                        ,first_channel
                        ,second_channel
                FROM    xyf_bi_dev.utm_source_channel_v1_cdf_v
                WHERE   pt = MAX_PT('xyf_bi_dev.utm_source_channel_v1_cdf_v')
                GROUP BY source
                         ,zero_channel
                         ,first_channel
                         ,second_channel
            ) qu_zhu
ON      qu_zhu.source = c.current_utm_source
LEFT JOIN   (
                SELECT  abbr
                        ,name
                FROM     -- xyf_ods.ods_sfy_sta_channel_df
xyf_dim.dim_sfy_sta_channel_df
                WHERE   pt = MAX_PT('xyf_dim.dim_sfy_sta_channel_df')
                GROUP BY abbr
                         ,name
            ) n
ON      c.current_utm_source = n.abbr --短信渠道商？
-- left JOIN (--实际执行人行完件评级？
-- )
LEFT JOIN   (
                SELECT  *
                FROM    xyf_dws.dws_inloan_user_order_df
                WHERE   pt = '${bizdate}'
                --AND     app IN ('xyf01') -- AND    date(first_order_time) >= '2024-01-01'
                AND     loan_flag = '首贷'
                and     business_line IN ('APP','小程序端')
            ) o
ON      o.cust_no = shouxin.cust_no
----是否预借款订单
LEFT JOIN   (
                            
                            SELECT  *
                                    ,GET_JSON_OBJECT(extend_data,'$.loanAmountFen') / 100 AS 预借款提交金额
                                    ,GET_JSON_OBJECT(extend_data,'$.term') AS 预借款提交期数
                            FROM    xyf_dwd.dwd_inloan_loan_pre_apply_df
                            WHERE   pt = MAX_PT('xyf_dwd.dwd_inloan_loan_pre_apply_df')
                        ) po
            ON      po.cust_no = c.cust_no
            AND     o.first_order_number = po.relate_order_no
LEFT JOIN   (
                SELECT  day_id_iso
                        ,CONCAT(day_week01_xf_new,'至',day_weekend_xf_new) AS week_range
                FROM    xyf_dim.dim_pub_date
            ) zhou
ON      TO_DATE(shouxin.created_time) = zhou.day_id_iso
--where b.biz_flow_number is null
GROUP BY shouxin.user_no
         ,date(shouxin.created_time)
         ,zhou.week_range
         ,SUBSTR(shouxin.created_time,1,7)
         ,case when upper(qu_zhu.source) like '%HRUI01%' then '其他付费-华瑞01'
        when upper(qu_zhu.source) like '%HRUI02%' then '其他付费-华瑞02' 
        else  qu_zhu.first_channel end 
         ,qu_gui.first_channel
         ,补件页是否可曝光 
         ,qu_zhu.second_channel
         ,qu_gui.second_channel
         ,是否半流程

;
DROP TABLE IF EXISTS xyf_bi_dev.bi_app_zhuanhua_wanjian_uid
---要加上归因渠道的数据，
;
CREATE table xyf_bi_dev.bi_app_zhuanhua_wanjian_uid AS 
select wanjian.user_no as wanjian_no, wanjian.dt as wanjian_dt
,qu_gui.first_channel 
,qu_gui.second_channel
,case when upper(qu_zhu.source) like '%HRUI01%' then '其他付费-华瑞01'
        when upper(qu_zhu.source) like '%HRUI02%' then '其他付费-华瑞02' 
        else  qu_zhu.first_channel end as zhu_first_channel
,qu_zhu.second_channel as zhu_second_channel
,zhou.week_range 
,case when RANDOMV3('fyvip_xk_test',wanjian.user_no,2) BETWEEN 50 and 99 then '飞跃可营销组'
        else '飞跃不可营销组'
        end as 是否飞跃可营销
,CASE   WHEN DATEDIFF(wanjian.dt,a.register_time) = 0 THEN '0d'
                WHEN DATEDIFF(wanjian.dt,a.register_time) BETWEEN 1 AND 30 THEN '1-30d'
                WHEN DATEDIFF(wanjian.dt,a.register_time) BETWEEN 31 AND 60 THEN '31d-60d'
                WHEN DATEDIFF(wanjian.dt,a.register_time) BETWEEN 61 AND 180 THEN '61d-180d'
                WHEN DATEDIFF(wanjian.dt,a.register_time) BETWEEN 181 AND 360 THEN '181d-360d'
                WHEN DATEDIFF(wanjian.dt,a.register_time) BETWEEN 361 AND 720 THEN '361d-720d'
                WHEN DATEDIFF(wanjian.dt,a.register_time) > 720 THEN '721d+'
                ELSE ''
        END AS 注册完件时间差
,case when jc.age <=24 THEN jc.age 
when jc.age between 55 and 60 then jc.age 
when jc.age between 25 and 54 then '25~54'
when jc.age >=61 then '61岁以上'
end as 年龄
,b.*
from 
(
select distinct shouxin.user_no,TO_DATE(shouxin.created_time) dt,shouxin.cust_no
from 
        (
            SELECT  *
                    ,ROW_NUMBER() OVER (PARTITION BY cust_no ORDER BY credit_success_time ) AS is_首次授信成功
                    ,CASE   WHEN client_code IN ('MPP001000068') THEN '小程序授信'
                            ELSE '非小程序授信'
                    END AS 是否小程序授信
            FROM    xyf_dwd.dwd_preloan_credit_apply_df
            WHERE   pt = '${bizdate}'
            AND     date(created_time) >= '2024-01-01'
            AND     app IN ('xyf01')
            AND     inner_app IN ( 'xyf01','xyf01_hrui02','xyf01_xcjr','xyf01_hrui01','xyf01_alyxy','xyf01_alygd','xyf01_alyfz','xyf01_zyxj01', 'xyf01_zyxjwld01','xyf01_zyxjzl01','xyf01_elm')
            and     nvl(app_activation_type,'') <> 'loan_recredit_activation'
        ) shouxin
 left anti join (
             SELECT  DISTINCT biz_flow_number
                FROM    xyf_dwd.dwd_inloan_t_decision_result_detail_df
                WHERE   enginecode = 'jcl_20240722000003'
                AND     pt = MAX_PT('xyf_dwd.dwd_inloan_t_decision_result_detail_df')
                AND     inner_app <> 'xyf01_test1'
        )b 
        on shouxin.biz_flow_number = b.biz_flow_number
        )wanjian
        left join xyf_bi.bi_app_zhuanhua_shouxin_uid b 
        on b.user_no = wanjian.user_no
        and b.dt = wanjian.dt
        LEFT JOIN (
  SELECT  id_card_number
                        ,cust_no
                        ,monthly_income
                        ,gender
                        ,age
                        ,education
                FROM    xyf_ads.ads_feature_idcard_essentialinfo_v2_df
                WHERE   pt = MAX_PT('xyf_ads.ads_feature_idcard_essentialinfo_v2_df')
)jc   
on wanjian.cust_no = jc.cust_no
        left join (
                 SELECT  c.*
                        ,CASE   WHEN a.attribution_source IS NULL THEN c.current_utm_source
                                ELSE a.attribution_source
                        END AS attribution_source
                FROM    (
                            SELECT  *
                            FROM    xyf_dws.dws_preloan_register_conversion_df
                            WHERE   pt = MAX_PT('xyf_dws.dws_preloan_register_conversion_df')
                            AND     app IN ('xyf','xyf01')
                        ) c
                LEFT JOIN   (
                                SELECT  user_id
                                        ,attribution_source
                                FROM    xyf_dwd.dwd_xyf_flow_sys_flow_attribution_result_dup_df
                                WHERE   pt = '${bizdate}'
                                QUALIFY ROW_NUMBER() OVER (PARTITION BY user_id ORDER BY created_time DESC ) = 1
                            ) a
                on c.app_user_id = a.user_id
        )a
        on a.app_user_id = wanjian.user_no
        LEFT JOIN   (
                SELECT  source
                        ,zero_channel
                        ,first_channel
                        ,second_channel
                FROM    xyf_bi_dev.utm_source_channel_v1_cdf_v
                WHERE   pt = MAX_PT('xyf_bi_dev.utm_source_channel_v1_cdf_v')
                GROUP BY source
                         ,zero_channel
                         ,first_channel
                         ,second_channel
            ) qu_gui
ON      qu_gui.source = a.attribution_source
LEFT JOIN   (
                SELECT  source
                        ,zero_channel
                        ,first_channel
                        ,second_channel
                FROM    xyf_bi_dev.utm_source_channel_v1_cdf_v
                WHERE   pt = MAX_PT('xyf_bi_dev.utm_source_channel_v1_cdf_v')
                GROUP BY source
                         ,zero_channel
                         ,first_channel
                         ,second_channel
            ) qu_zhu
ON      qu_zhu.source = a.current_utm_source
LEFT JOIN   (
                SELECT  day_id_iso
                        ,CONCAT(day_week01_xf_new,'至',day_weekend_xf_new) AS week_range
                FROM    xyf_dim.dim_pub_date
            ) zhou
ON      TO_DATE(wanjian.dt) = zhou.day_id_iso
        


;

DROP TABLE IF EXISTS bi_app_zhuanhua_shouxin_sum
;

CREATE TABLE bi_app_zhuanhua_shouxin_sum AS
SELECT  wanjian_dt as dt
        ,week_range as wt
        ,substr(wanjian_dt,1,7) as mt 
        ,授信金额_level
        ---,biz_type_bi
        ,zhu_first_channel as biz_type_bi
        ,zhu_second_channel as biz_type_bi_2
        ,first_channel as biz_type_bi_3 
        ,second_channel as biz_type_bi_4
        ,实际执行人行完件评级_v3_v4 
        ,年龄
        ,是否半流程
        ,注册完件时间差 as 注册授信时间差
        ,CASE   WHEN 是否小程序授信 >= 1 THEN '小程序授信'
                ELSE '否'
        END AS 是否小程序授信
        ,CASE   WHEN 是否首次APP授信成功 = 1 THEN '是'
                ELSE '否'
        END AS 是否首次APP授信成功
        ,case when 是否通过预借款发起 = 1 then    '预借款发起'
        else '未预借款发起'
        end as 是否通过预借款发起
        ,补件页是否可曝光 ---
        ,是否飞跃可营销
        ,点击借款按钮 as T0是否点击借款按钮
        ,COUNT(distinct wanjian_no) 完件人数
        ,COUNT(DISTINCT user_no) 授信通过
        ,SUM(授信金额) 授信额度_shu
        ,sum(T0登录)T0登录
        ,sum(进入提现页)进入提现页
        ,sum(点击借款按钮)点击借款按钮
        ,sum(点击借款按钮_无需补件)点击借款按钮_无需补件---
        ,sum(点击借款按钮_不缺三要素)点击借款按钮_不缺三要素---
        ,sum(进入补件页人数)进入补件页人数
        ,sum(进入补件页完成补件人数)进入补件页完成补件人数---
        ,sum(需要补件用户数)需要补件用户数
        ,sum(需要补联系人用户)需要补联系人用户
        ,sum(需要补OCR用户)需要补OCR用户
        ,sum(需要补银行卡用户)需要补银行卡用户
        ,sum(补充联系人)补充联系人
        ,sum(补充OCR)补充OCR
        ,sum(补充银行卡用户)补充银行卡用户
        ,sum(完成所有补件用户)完成所有补件用户 
        ,SUM(提现人) 提现人
        ,SUM(T3提现人) T3提现人
        ,SUM(T7提现人) T7提现人
        ,SUM(T30提现人) T30提现人
        ,SUM(提现金额) 提现金额
        ,SUM(提现金额7) 提现金额7
        ,SUM(提现金额_PERIOD) 提现金额_PERIOD
        ,SUM(放款金额) 放款金额
        ,SUM(T3放款金额) T3放款金额
        ,SUM(T7放款金额) T7放款金额
        ,SUM(放款金额_PERIOD) 放款金额_PERIOD
        ,SUM(风控通过人) 风控通过人
        ,SUM(T3风控通过人) T3风控通过人
        ,SUM(T7风控通过人) T7风控通过人
        ----风控通过金额,T7风控通过金额
        ,SUM(风控通过金额)风控通过金额
        ,SUM(T7风控通过金额) T7风控通过金额
        ,SUM(放款人) 资方通过人
        ,SUM(T3放款人) T3资方通过人
        ,SUM(T7放款人) T7资方通过人
        ,SUM(放款人) 放款人
        ,SUM(T3放款人) T3放款人
        ,SUM(T7放款人) T7放款人
        ,SUM(T30放款人) T30放款人
        ,SUM(授信额度_放款) 授信额度_放款
        ,SUM(授信额度_放款7) 授信额度_放款7
        ,SUM(授信额度_放款30) 授信额度_放款30
--FROM    xyf_bi.bi_app_zhuanhua_shouxin_uid 
from xyf_bi_dev.bi_app_zhuanhua_wanjian_uid
GROUP BY wanjian_dt
--dt
         --,wt
         ,week_range
         ,mt 
         ,年龄
         --,biz_type_bi
         --,biz_type_bi_3 
         ,zhu_first_channel 
        ,zhu_second_channel 
        ,second_channel
         ,first_channel
         ,授信金额_level
         ,实际执行人行完件评级_v3_v4
         ,注册完件时间差
         ,CASE   WHEN 是否小程序授信 >= 1 THEN '小程序授信'
                 ELSE '否'
         END
         ,CASE   WHEN 是否首次APP授信成功 = 1 THEN '是'
                 ELSE '否'
         END
         ,是否通过预借款发起
         ,补件页是否可曝光 
         ,T0是否点击借款按钮
         ,是否飞跃可营销
         ,是否半流程
;

DROP TABLE IF EXISTS xyf_bi_dev.bi_app_zhuanhua_shouxin_syt_new
;

CREATE TABLE xyf_bi_dev.bi_app_zhuanhua_shouxin_syt_new AS
SELECT  dt
        ,wt
        ,mt
        ,授信金额_level AS credit_amt_level
        ,biz_type_bi
        ,biz_type_bi_3
        ,实际执行人行完件评级_v3_v4 AS actual_rh_wanjian_v3_v4
        ,注册授信时间差 AS reg_loan_diff_time
        ,是否小程序授信 AS is_xcx_credit
        ,是否飞跃可营销 as is_feiyue_mkt
        ,是否首次app授信成功 AS is_xcx_first_credit_success
        ,授信通过 AS credit_success_cnt
        ,年龄 as age
        ,授信额度_shu AS credit_amt
        ,提现人 AS order_apply_cnt
        ,t3提现人 AS order_apply3_cnt
        ,t7提现人 AS order_apply7_cnt
        ,t30提现人 AS order_apply30_cnt
        ,提现金额 AS apply_amt
        ,提现金额7 AS apply_amt7 --new
        ,提现金额_period AS apply_amt_cross_period
        ,放款金额 AS loan_amt
        ,t3放款金额 AS loan3_amt
        ,t7放款金额 AS loan7_amt
        ,放款金额_period AS loam_amt_cross_period
        ,风控通过人 AS risk_pass_cnt
        ,t3风控通过人 AS risk_pass3_cnt
        ,t7风控通过人 AS risk_pass7_cnt
        ----风控通过金额,T7风控通过金额
        ,风控通过金额 AS risk_pass_amt---
        ,T7风控通过金额 AS risk_pass7_amt---
        ,资方通过人 AS remit_pass_cnt
        ,t3资方通过人 AS remit_pass3_cnt
        ,t7资方通过人 AS remit_pass7_cnt
        ,放款人 AS loan_cnt
        ,t3放款人 AS loan3_cnt
        ,t7放款人 AS loan7_cnt
        ,t30放款人 AS loan30_cnt
        ,授信额度_放款 AS credit_amt_if_loan
        ,授信额度_放款7 AS credit_amt_if_loan7
        ,授信额度_放款30 AS credit_amt_if_loan30
        ,完件人数 AS wanjian_cnt
        ,进入提现页 AS  view_order_page_cnt
        ,点击借款按钮 AS click_apply_botton
        ,需要补件用户数 AS need_add_info_cnt
        ,需要补联系人用户 AS need_add_contact_cnt
        ,需要补OCR用户 AS need_add_ocr_cnt
        ,需要补银行卡用户 AS need_add_bank_cnt
        ,补充联系人 as add_contact_cnt
        ,补充OCR as add_ocr_cnt
        ,补充银行卡用户 as add_bank_cnt
        ,完成所有补件用户 as add_info_cnt
        ,进入补件页人数 as view_add_page_cnt
        ,是否通过预借款发起 as is_pre_apply_order
        ,进入补件页完成补件人数 as view_and_finish_info_cnt
        ,T0是否点击借款按钮 as T0_click_apply_botton
        ,是否半流程 as is_halft_api
FROM    xyf_bi.bi_app_zhuanhua_shouxin_sum
